# ECON 662 — Problem Set 2
## Part (c.1): Forward Simulation MLE Using Equation (5)

Estimate $(\theta,\rho,\sigma_\rho)$ using forward simulation to approximate the value function, instead of:
- solving a Bellman fixed point as in Part (a), or
- using the Hotz-Miller matrix inversion as in Part (b).

Following the homework notes and Lecture 15, the first two steps are the same as Part (b):

1. Estimate the replacement-cost process and the state transition objects.
2. Estimate the conditional choice probabilities (CCPs) from the data.

Then, for each state $s=(x,RC)$, approximate the value function by forward simulation using Equation (5):

$$
\hat V(s;\theta)
=
\frac{1}{N_{\text{sim}}}
\sum_{n=1}^{N_{\text{sim}}}
\sum_{\tau=0}^{T_{\text{sim}}-1}
\beta^\tau
\Bigl[
\bar u(s_{n\tau},a_{n\tau};\theta)
\;+\;
\gamma
\;-\;
\ln \hat P(a_{n\tau}\mid s_{n\tau})
\Bigr],
$$

where
- $a_{n\tau}$ is drawn from the estimated CCP $\hat P(\cdot\mid s_{n\tau})$,
- $s_{n,\tau+1}$ is drawn from the estimated transition process,
- $\bar u$ is the deterministic payoff component,
- $\gamma \approx 0.5772$ is Euler's constant.

#### Import Libraries

In [1]:
import time
import numpy as np
import pandas as pd
from scipy.optimize import minimize

np.random.seed(42)

#### Load Data


In [2]:
# Variables:
# i  = bus id
# t  = month
# a  = action (0 = maintain, 1 = replace)
# x  = mileage state in {1, ..., 7}
# rc = replacement cost (continuous)
df = pd.read_csv("ddc_pset.csv")
df = df.sort_values(["i", "t"]).reset_index(drop=True)

print(f"Loaded {len(df):,} observations for {df['i'].nunique():,} buses.")
print(f"RC range: [{df['rc'].min():.4f}, {df['rc'].max():.4f}]")
print(f"Replacement rate: {df['a'].mean():.4f}")
df.head(10)


Loaded 100,000 observations for 1,000 buses.
RC range: [31.5000, 62.5000]
Replacement rate: 0.1724


,i,t,a,x,rc
0,1,1,0,4,45.0
1,1,2,0,5,49.0
2,1,3,0,6,54.0
3,1,4,1,7,47.0
4,1,5,0,1,56.0
5,1,6,0,2,52.5
6,1,7,0,3,55.0
7,1,8,0,4,50.5
8,1,9,0,5,52.0
9,1,10,0,6,42.0


#### Global Constants


In [3]:
BETA        = 0.95
N_X         = 7
N_RC_BINS   = 8
S           = N_X * N_RC_BINS
GAMMA       = 0.5772156649

# Forward-simulation tuning parameters for Part (c.1)
N_SIM       = 100
T_SIM       = 120
SIM_SEED    = 12345

print(f"beta        = {BETA}")
print(f"mileage x   = {N_X} states")
print(f"RC bins     = {N_RC_BINS}")
print(f"|S|         = {S}")
print(f"N_SIM       = {N_SIM}")
print(f"T_SIM       = {T_SIM}")
print(f"beta^T_SIM  = {BETA**T_SIM:.6f}")


beta        = 0.95
mileage x   = 7 states
RC bins     = 8
|S|         = 56
N_SIM       = 100
T_SIM       = 120
beta^T_SIM  = 0.002122


### Estimate the Transition Objects and $(\rho_0,\rho_1,\sigma_\rho)$

This step is the same as Part (b).

We first discretize observed replacement costs into equal-width bins:

$$
RC \in [RC_{\min}, RC_{\max}] \longrightarrow \text{8 empirical bins}.
$$

Let $\Pi_{j,j'}$ be the empirical probability that $RC$ moves from bin $j$ today to bin $j'$ tomorrow.
This gives us the exogenous transition for the replacement-cost state.

Separately, we estimate the continuous AR(1) process

$$
RC_t = \rho_0 + \rho_1 RC_{t-1} + e_t, \qquad e_t \sim N(0,\sigma_\rho^2),
$$

by OLS using the observed continuous $RC$ values.

As in Parts (a) and (b), the Gaussian likelihood for the AR(1) process is separable from the choice likelihood, so we estimate $(\rho_0,\rho_1,\sigma_\rho)$ once before the MLE over $\theta$.


In [4]:
# Equal-width bins over the observed RC support
rc_min = df['rc'].min()
rc_max = df['rc'].max()
bin_edges = np.linspace(rc_min, rc_max, N_RC_BINS + 1)
rc_grid = (bin_edges[:-1] + bin_edges[1:]) / 2.0

# Assign each observation to a bin (0-indexed)
rc_bin_idx = np.searchsorted(bin_edges[1:], df['rc'].values, side='left')
rc_bin_idx = np.clip(rc_bin_idx, 0, N_RC_BINS - 1)
df['rc_bin'] = rc_bin_idx
df['rc_mid'] = rc_grid[rc_bin_idx]

print("RC bin edges:")
print(np.round(bin_edges, 3))
print("\nRC grid (bin midpoints):")
print(np.round(rc_grid, 3))

# Empirical RC transition matrix Pi
C = np.zeros((N_RC_BINS, N_RC_BINS))
for _, g in df.groupby("i"):
    bins = g['rc_bin'].values
    for j_from, j_to in zip(bins[:-1], bins[1:]):
        C[j_from, j_to] += 1

Pi = C / C.sum(axis=1, keepdims=True)

print("\nEmpirical RC transition matrix Pi:")
print(np.round(Pi, 4))
print("Row sums:")
print(np.round(Pi.sum(axis=1), 10))

# OLS for the continuous AR(1): RC_t = rho0 + rho1 * RC_{t-1} + e_t
rc_prev_list, rc_curr_list = [], []
for _, g in df.groupby("i"):
    vals = g['rc'].values
    rc_prev_list.append(vals[:-1])
    rc_curr_list.append(vals[1:])

rc_prev = np.concatenate(rc_prev_list)
rc_curr = np.concatenate(rc_curr_list)

X_ols = np.column_stack([np.ones(len(rc_prev)), rc_prev])
beta_ols = np.linalg.solve(X_ols.T @ X_ols, X_ols.T @ rc_curr)
rho0_hat, rho1_hat = beta_ols
sigma_rho_hat = (rc_curr - X_ols @ beta_ols).std(ddof=1)

print("\nAR(1) estimates from continuous RC:")
print(f"rho0_hat      = {rho0_hat:.4f}")
print(f"rho1_hat      = {rho1_hat:.4f}")
print(f"sigma_rho_hat = {sigma_rho_hat:.4f}")


RC bin edges:
[31.5   35.375 39.25  43.125 47.    50.875 54.75  58.625 62.5  ]

RC grid (bin midpoints):
[33.438 37.312 41.188 45.062 48.938 52.812 56.688 60.562]

Empirical RC transition matrix Pi:
[[0.     0.3333 0.6667 0.     0.     0.     0.     0.    ]
 [0.2    0.4    0.2    0.2    0.     0.     0.     0.    ]
 [0.1333 0.     0.4    0.2    0.2    0.0667 0.     0.    ]
 [0.     0.087  0.087  0.3043 0.3913 0.087  0.0435 0.    ]
 [0.     0.     0.1154 0.2692 0.2692 0.3077 0.0385 0.    ]
 [0.     0.     0.0556 0.2778 0.2778 0.1111 0.2222 0.0556]
 [0.     0.     0.     0.     0.2857 0.5714 0.     0.1429]
 [0.     0.     0.     0.     0.     0.5    0.5    0.    ]]
Row sums:
[1. 1. 1. 1. 1. 1. 1. 1.]

AR(1) estimates from continuous RC:
rho0_hat      = 17.8269
rho1_hat      = 0.6249
sigma_rho_hat = 4.6060


### Build Full State Transitions and Estimate CCPs

The combined state is

$$
s = (x, j),
$$

where $x \in \{1,\dots,7\}$ is mileage and $j \in \{0,\dots,7\}$ is the RC bin.
Hence there are

$$
|S| = 7 \times 8 = 56
$$

discrete states.

The action-specific state transitions are the same as in Part (b):

- If $a=0$ (maintain), then $x' = \min(x+1,7)$.
- If $a=1$ (replace), then $x' = 1$.
- In both cases, the RC-bin transition is governed by the empirical matrix $\Pi$.

We also estimate the CCP from observed state frequencies:

$$
\hat P(a=1\mid s) = \frac{N_1(s)}{N_0(s) + N_1(s)},
\qquad
\hat P(a=0\mid s) = 1 - \hat P(a=1\mid s).
$$

To avoid $\log(0)$ in Equation (5), we clip these probabilities slightly away from 0 and 1.


In [5]:
# Flat state indexing: s = xi * N_RC_BINS + j
xi_of_s = np.array([xi for xi in range(N_X) for _ in range(N_RC_BINS)])
j_of_s  = np.array([j  for _ in range(N_X) for j in range(N_RC_BINS)])

x_of_s  = xi_of_s + 1
rc_of_s = rc_grid[j_of_s]

# Next mileage index after each action
xi_next_mnt = np.minimum(xi_of_s + 1, N_X - 1)
xi_next_rep = np.zeros(S, dtype=int)

# Full action-specific transition matrices TP0 and TP1, shape (56, 56)
TP0 = np.zeros((S, S))
TP1 = np.zeros((S, S))

for s in range(S):
    j = j_of_s[s]

    col0 = xi_next_mnt[s] * N_RC_BINS
    TP0[s, col0:col0 + N_RC_BINS] = Pi[j, :]

    col1 = xi_next_rep[s] * N_RC_BINS
    TP1[s, col1:col1 + N_RC_BINS] = Pi[j, :]

print(f"TP0 shape: {TP0.shape}")
print(f"TP1 shape: {TP1.shape}")
print(f"TP0 row sums: [{TP0.sum(1).min():.8f}, {TP0.sum(1).max():.8f}]")
print(f"TP1 row sums: [{TP1.sum(1).min():.8f}, {TP1.sum(1).max():.8f}]")

# State-level counts
N1 = np.zeros((N_X, N_RC_BINS))
N0 = np.zeros((N_X, N_RC_BINS))

for xi_obs, j_obs, a_obs in zip(df['x'].values.astype(int) - 1,
                                df['rc_bin'].values.astype(int),
                                df['a'].values.astype(int)):
    if a_obs == 1:
        N1[xi_obs, j_obs] += 1
    else:
        N0[xi_obs, j_obs] += 1

N_total = N0 + N1

# Smoothed/clipped CCP
eps_ccp = 1e-6
P1_hat = np.where(N_total > 0, N1 / N_total, eps_ccp)
P1_hat = np.clip(P1_hat, eps_ccp, 1.0 - eps_ccp)
P0_hat = 1.0 - P1_hat

P1_flat = P1_hat.ravel()
P0_flat = P0_hat.ravel()

print("\nEstimated CCP  P_hat(a=1 | x, rc_bin):")
print("       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_hat[xi], 3)}")

print(f"\nTotal observations recovered from N0+N1: {N_total.sum():.0f}")


TP0 shape: (56, 56)
TP1 shape: (56, 56)
TP0 row sums: [1.00000000, 1.00000000]
TP1 row sums: [1.00000000, 1.00000000]

Estimated CCP  P_hat(a=1 | x, rc_bin):
       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7
  x=1: [0.028 0.019 0.009 0.005 0.004 0.002 0.    0.   ]
  x=2: [0.078 0.066 0.03  0.022 0.015 0.008 0.004 0.008]
  x=3: [0.265 0.173 0.134 0.075 0.049 0.039 0.013 0.011]
  x=4: [0.439 0.343 0.277 0.171 0.128 0.086 0.067 0.025]
  x=5: [0.621 0.531 0.43  0.342 0.252 0.187 0.134 0.082]
  x=6: [0.669 0.63  0.565 0.471 0.369 0.312 0.225 0.091]
  x=7: [0.826 0.74  0.643 0.565 0.478 0.397 0.28  0.208]

Total observations recovered from N0+N1: 100000


### Forward Simulation Approximation of $\hat V(s;\theta)$

For Part (c.1), we follow Equation (5) literally:

$$
\hat V(s;\theta)
=
\frac{1}{N_{\text{sim}}}
\sum_{n=1}^{N_{\text{sim}}}
\sum_{\tau=0}^{T_{\text{sim}}-1}
\beta^\tau
\Bigl[
\bar u(s_{n\tau},a_{n\tau};\theta)
\;+\;
\gamma - \ln \hat P(a_{n\tau}\mid s_{n\tau})
\Bigr].
$$

At each simulated period:

1. Draw an action from the estimated CCP.
2. Draw the next state from the estimated transition matrix conditional on that action.
3. Add the deterministic flow utility plus the Type-I-EV correction term $\gamma - \ln \hat P(a\mid s)$.

The deterministic payoff is

$$
\bar u(s,a;\theta)=
\begin{cases}
-\theta_1 x, & a=0, \\\\
-\theta_2 RC, & a=1.
\end{cases}
$$

Because the CCPs and transition matrices are fixed from the first-stage estimation, we can use common random numbers and simulate all paths once outside the optimizer. Then the simulated value function is linear in $(\theta_1,\theta_2)$:

$$
\hat V(s;\theta)
=
-\theta_1 \cdot M_s
-\theta_2 \cdot R_s
+ K_s,
$$

where

- $M_s$ is the simulated discounted average of $x_\tau \mathbf{1}\{a_\tau=0\}$,
- $R_s$ is the simulated discounted average of $RC_\tau \mathbf{1}\{a_\tau=1\}$,
- $K_s$ is the simulated discounted average of $\gamma - \ln \hat P(a_\tau\mid s_\tau)$.

This keeps the optimizer deterministic and much faster while staying fully faithful to Equation (5).

In [6]:
discounts = BETA ** np.arange(T_SIM)
cumTP0 = np.cumsum(TP0, axis=1)
cumTP1 = np.cumsum(TP1, axis=1)

# Numerical guard: force the last cumulative entry to be exactly 1
cumTP0[:, -1] = 1.0
cumTP1[:, -1] = 1.0


def simulate_value_components(P1_flat, P0_flat, x_of_s, rc_of_s, cumTP0, cumTP1,
                              n_sim=N_SIM, t_sim=T_SIM, seed=SIM_SEED):
    '''
    Pre-simulate the objects in Equation (5) using common random numbers.

    Returns three length-S vectors:
      maint_component[s] = E_sim[sum beta^t * x_t * 1{a_t=0} | s0=s]
      repl_component[s]  = E_sim[sum beta^t * rc_t * 1{a_t=1} | s0=s]
      corr_component[s]  = E_sim[sum beta^t * (gamma - log P_hat(a_t|s_t)) | s0=s]
    '''
    rng = np.random.default_rng(seed)

    maint_component = np.zeros(S)
    repl_component  = np.zeros(S)
    corr_component  = np.zeros(S)

    for s0 in range(S):
        states = np.full(n_sim, s0, dtype=int)

        maint_acc = np.zeros(n_sim)
        repl_acc  = np.zeros(n_sim)
        corr_acc  = np.zeros(n_sim)

        for tau in range(t_sim):
            beta_tau = discounts[tau]

            p1 = P1_flat[states]
            p0 = P0_flat[states]

            u_action = rng.random(n_sim)
            a1 = u_action < p1
            a0 = ~a1

            p_selected = np.where(a1, p1, p0)

            maint_acc += beta_tau * x_of_s[states] * a0
            repl_acc  += beta_tau * rc_of_s[states] * a1
            corr_acc  += beta_tau * (GAMMA - np.log(p_selected))

            u_next = rng.random(n_sim)
            cdf = np.where(a1[:, None], cumTP1[states, :], cumTP0[states, :])
            next_states = (u_next[:, None] > cdf).sum(axis=1)
            next_states = np.minimum(next_states, S - 1)
            states = next_states

        maint_component[s0] = maint_acc.mean()
        repl_component[s0]  = repl_acc.mean()
        corr_component[s0]  = corr_acc.mean()

    return maint_component, repl_component, corr_component


t0_sim = time.time()
maint_component, repl_component, corr_component = simulate_value_components(
    P1_flat=P1_flat,
    P0_flat=P0_flat,
    x_of_s=x_of_s,
    rc_of_s=rc_of_s,
    cumTP0=cumTP0,
    cumTP1=cumTP1,
    n_sim=N_SIM,
    t_sim=T_SIM,
    seed=SIM_SEED,
)
t_simulation = time.time() - t0_sim

print(f"Forward simulation pre-computation finished in {t_simulation:.2f} seconds.")
print(f"maint_component shape: {maint_component.shape}")
print(f"repl_component  shape: {repl_component.shape}")
print(f"corr_component  shape: {corr_component.shape}")

print("\nFirst 10 entries of simulated components:")
print("maint_component:", np.round(maint_component[:10], 3))
print("repl_component :", np.round(repl_component[:10], 3))
print("corr_component :", np.round(corr_component[:10], 3))


Forward simulation pre-computation finished in 0.24 seconds.
maint_component shape: (56,)
repl_component  shape: (56,)
corr_component  shape: (56,)

First 10 entries of simulated components:
maint_component: [50.174 50.543 51.643 51.731 52.566 51.359 51.133 52.636 53.143 52.681]
repl_component : [145.679 141.849 139.79  140.381 139.478 142.39  143.089 140.171 147.691
 149.059]
corr_component : [18.124 17.783 17.523 17.741 17.764 17.752 17.761 17.6   18.406 18.373]


### From Simulated $\hat V(s;\theta)$ to Model CCPs and the Likelihood

Once we have $\hat V(s;\theta)$, we compute the continuation value for each action:

$$
EV(s,0;\theta) = \sum_{s'} TP_0(s,s') \hat V(s';\theta),
\qquad
EV(s,1;\theta) = \sum_{s'} TP_1(s,s') \hat V(s';\theta).
$$

Then the choice-specific values are

$$
\delta_0(s;\theta) = -\theta_1 x_s + \beta EV(s,0;\theta),
$$

$$
\delta_1(s;\theta) = -\theta_2 RC_s + \beta EV(s,1;\theta).
$$

Under Type I EV shocks, the model-implied replacement probability is

$$
\tilde P(a=1\mid s;\theta)
=
\frac{\exp(\delta_1(s;\theta))}
{\exp(\delta_0(s;\theta)) + \exp(\delta_1(s;\theta))}.
$$

Finally, we maximize the same state-level MLE objective as in Parts (a) and (b):

$$
\ell(\theta_1,\theta_2)
=
\sum_{s=1}^{|S|}
\Bigl[
N_1(s)\ln \tilde P(a=1\mid s;\theta)
+
N_0(s)\ln \tilde P(a=0\mid s;\theta)
\Bigr].
$$


In [7]:
def simulated_value(theta1, theta2):
    '''
    Equation (5) value approximation, using the pre-simulated components:

      V_hat(s; theta) = -theta1 * maint_component[s]
                        -theta2 * repl_component[s]
                        + corr_component[s]
    '''
    return -theta1 * maint_component - theta2 * repl_component + corr_component


def model_ccp(theta1, theta2):
    '''
    From V_hat(s;theta), compute continuation values EV(s,a;theta),
    then recover model-implied CCPs from the logit formula.
    '''
    V_hat = simulated_value(theta1, theta2)

    EV0 = TP0 @ V_hat
    EV1 = TP1 @ V_hat

    delta0 = -theta1 * x_of_s  + BETA * EV0
    delta1 = -theta2 * rc_of_s + BETA * EV1

    dmax = np.maximum(delta0, delta1)
    exp0 = np.exp(delta0 - dmax)
    exp1 = np.exp(delta1 - dmax)
    P1_model = exp1 / (exp0 + exp1)

    return V_hat, P1_model.reshape(N_X, N_RC_BINS)


def neg_log_likelihood(params):
    theta1, theta2 = params

    if theta1 <= 0 or theta2 <= 0:
        return 1e10

    _, P1_model = model_ccp(theta1, theta2)
    P0_model = 1.0 - P1_model

    eps = 1e-12
    ll = np.sum(N1 * np.log(P1_model + eps) + N0 * np.log(P0_model + eps))
    return -ll


theta_init = np.array([1.0, 1.0])
V_init, P1_init = model_ccp(theta_init[0], theta_init[1])

print("Initial-guess diagnostics at theta1=1, theta2=1")
print(f"NLL(theta_init) = {neg_log_likelihood(theta_init):.4f}")
print("\nFirst 10 entries of V_hat(theta_init):")
print(np.round(V_init[:10], 3))

print("\nModel-implied P(a=1|x,rc_bin) at theta1=theta2=1:")
print("       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_init[xi], 3)}")


Initial-guess diagnostics at theta1=1, theta2=1
NLL(theta_init) = 202947.8948

First 10 entries of V_hat(theta_init):
[-177.729 -174.608 -173.911 -174.371 -174.28  -175.996 -176.461 -175.208
 -182.428 -183.367]

Model-implied P(a=1|x,rc_bin) at theta1=theta2=1:
       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7
  x=1: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=2: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=3: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=4: [0.016 0.    0.    0.    0.    0.    0.    0.   ]
  x=5: [0.069 0.    0.    0.    0.    0.    0.    0.   ]
  x=6: [0.812 0.012 0.005 0.    0.    0.    0.    0.   ]
  x=7: [0.922 0.033 0.013 0.001 0.    0.    0.    0.   ]


### Optimization

The forward simulation has already been done outside the optimizer.
So each optimizer call now does only three things:

1. Construct $\hat V(s;\theta)$ from the simulated components.
2. Compute continuation values and model-implied CCPs.
3. Evaluate the state-level log-likelihood.

For comparability with Parts (a) and (b), we again use Nelder-Mead.

In [8]:
print("--- Part (c.1): Forward Simulation MLE ---")
print(f"Initial theta = {theta_init}")
print(f"Initial NLL   = {neg_log_likelihood(theta_init):.4f}\n")

t0_theta = time.perf_counter()
result = minimize(
    neg_log_likelihood,
    theta_init,
    method="Nelder-Mead",
    options=dict(maxiter=5000, xatol=1e-8, fatol=1e-8, disp=True),
)
optimizer_runtime_sec = time.perf_counter() - t0_theta
optimizer_runtime_ms = 1000.0 * optimizer_runtime_sec
running_time_sec = t_simulation + optimizer_runtime_sec
running_time_ms = 1000.0 * running_time_sec

print(f"\nOptimizer-only time: {optimizer_runtime_sec:.4f} seconds ({optimizer_runtime_ms:.2f} ms)")
print(f"Comparable running time: {running_time_sec:.4f} seconds ({running_time_ms:.2f} ms)")


--- Part (c.1): Forward Simulation MLE ---
Initial theta = [1. 1.]
Initial NLL   = 202947.8948

Optimization terminated successfully.
         Current function value: 34001.052261
         Iterations: 77
         Function evaluations: 151

Optimizer-only time: 0.0035 seconds (3.48 ms)
Comparable running time: 0.2389 seconds (238.91 ms)


### Results


In [9]:
theta1_hat, theta2_hat = result.x
V_hat_final, P1_final = model_ccp(theta1_hat, theta2_hat)

print("=" * 62)
print("     PART (c.1) FORWARD SIMULATION -- MLE RESULTS")
print("=" * 62)
print(f"theta1_hat    (mileage cost coeff.)     = {theta1_hat:.4f}")
print(f"theta2_hat    (replacement cost coeff.) = {theta2_hat:.4f}")
print("-" * 62)
print("Pre-estimated RC process parameters:")
print(f"rho0_hat                                  = {rho0_hat:.4f}")
print(f"rho1_hat                                  = {rho1_hat:.4f}")
print(f"sigma_rho_hat                             = {sigma_rho_hat:.4f}")
print("-" * 62)
print(f"Choice log-likelihood                     = {-result.fun:.4f}")
print(f"Converged                                 = {result.success}")
print(f"Optimizer iterations                      = {result.nit}")
print(f"Forward-simulation precomputation time    = {t_simulation:.4f} sec")
print(f"Optimizer-only time                       = {optimizer_runtime_sec:.4f} sec ({optimizer_runtime_ms:.2f} ms)")
print(f"Comparable running time                   = {running_time_sec:.4f} sec ({running_time_ms:.2f} ms)")
print("=" * 62)

print("\nEstimated model CCP  P(a=1 | x, rc_bin):")
print("       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_final[xi], 3)}")


     PART (c.1) FORWARD SIMULATION -- MLE RESULTS
theta1_hat    (mileage cost coeff.)     = 0.3955
theta2_hat    (replacement cost coeff.) = 0.1537
--------------------------------------------------------------
Pre-estimated RC process parameters:
rho0_hat                                  = 17.8269
rho1_hat                                  = 0.6249
sigma_rho_hat                             = 4.6060
--------------------------------------------------------------
Choice log-likelihood                     = -34001.0523
Converged                                 = True
Optimizer iterations                      = 77
Forward-simulation precomputation time    = 0.2354 sec
Optimizer-only time                       = 0.0035 sec (3.48 ms)
Comparable running time                   = 0.2389 sec (238.91 ms)

Estimated model CCP  P(a=1 | x, rc_bin):
       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7
  x=1: [0.028 0.017 0.009 0.006 0.003 0.002 0.001 0.001]
  x=2: [0.091 0.057 0.036 0.026 0.015

### Summary

The first two steps are unchanged from Part (b):

1. Estimate the RC transition objects and the AR(1) parameters.
2. Estimate the CCP from the data.

The only change is Step 3:

- Part (b): use the Hotz-Miller linear inversion $AV=b(\theta)$.
- Part (c.1): use forward simulation and Equation (5) to approximate $\hat V(s;\theta)$.

Then, just as before, we:

1. Compute $EV(s,a;\theta)$ from the transition matrix and $\hat V$.
2. Recover model-implied choice probabilities from the logit formula.
3. Form the state-level MLE objective.
